In [3]:
import pandas as pd

df = pd.read_csv('../data/processed/features.csv') 
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

print(df['Date'].min(), '->', df['Date'].max())
print(df.shape)

1999-02-22 00:00:00 -> 2026-08-19 00:00:00
(10041, 86)


In [4]:
train_end = '2020-12-31'
val_end   = '2023-12-31'
# test = everything after that, up to dataset's last date

train_df = df[df['Date'] <= train_end].reset_index(drop=True)
val_df   = df[(df['Date'] > train_end) & (df['Date'] <= val_end)].reset_index(drop=True)
test_df  = df[df['Date'] > val_end].reset_index(drop=True)

print("Train:", train_df['Date'].min(), "->", train_df['Date'].max(), "| rows:", len(train_df))
print("Val:  ", val_df['Date'].min(), "->", val_df['Date'].max(), "| rows:", len(val_df))
print("Test: ", test_df['Date'].min(), "->", test_df['Date'].max(), "| rows:", len(test_df))

Train: 1999-02-22 00:00:00 -> 2020-12-31 00:00:00 | rows: 7984
Val:   2021-01-01 00:00:00 -> 2023-12-31 00:00:00 | rows: 1095
Test:  2024-01-01 00:00:00 -> 2026-08-19 00:00:00 | rows: 962


In [5]:
# 1. No overlap between splits
assert len(set(train_df['Date']) & set(val_df['Date'])) == 0
assert len(set(val_df['Date']) & set(test_df['Date'])) == 0

# 2. Train's max date should always be before val's min date, and so on
assert train_df['Date'].max() < val_df['Date'].min()
assert val_df['Date'].max() < test_df['Date'].min()

# 3. Check class balance in each split — imbalance ratio zyada different nahi hona chahiye
for name, split in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f"{name} positive rate: {split['target_flare_MX_occurred'].mean():.2%}")

train positive rate: 14.85%
val positive rate: 25.94%
test positive rate: 49.48%


In [6]:
target_cols = ['target_flare_MX_occurred', 'target_flare_MX_count']
drop_cols = ['Date'] + target_cols   # Date model ko feature ke roop me nahi jaana chahiye

X_train = train_df.drop(columns=drop_cols)
y_train = train_df['target_flare_MX_occurred']

X_val = val_df.drop(columns=drop_cols)
y_val = val_df['target_flare_MX_occurred']

X_test = test_df.drop(columns=drop_cols)
y_test = test_df['target_flare_MX_occurred']

In [7]:
train_df.to_csv('../data/processed/train.csv', index=False)
val_df.to_csv('../data/processed/val.csv', index=False)
test_df.to_csv('../data/processed/test.csv', index=False)

## Flare Class Distribution Across Splits

Check karte hain X, M, S class flares kaise distribute hue hain train/val/test me — solar cycle ke peak/quiet periods ki wajah se class imbalance split ke hisaab se badal sakta hai.

In [9]:
cols = ['Flares_X', 'Flares_M', 'Flares_S']

summary = []
for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    for c in cols:
        total = split[c].sum()
        days_with = (split[c] > 0).sum()
        pct_days = days_with / len(split) * 100
        summary.append({
            'Split': name,
            'Flare_Class': c,
            'Total_Count': int(total),
            'Days_With_Flare': days_with,
            'Percent_of_Days': round(pct_days, 1)
        })

summary_df = pd.DataFrame(summary)
summary_df

,Split,Flare_Class,Total_Count,Days_With_Flare,Percent_of_Days
0,Train,Flares_X,157,135,1.7
1,Train,Flares_M,2059,1136,14.2
2,Train,Flares_S,25572,3969,49.7
3,Val,Flares_X,22,22,2.0
4,Val,Flares_M,563,279,25.5
5,Val,Flares_S,5340,766,70.0
6,Test,Flares_X,85,66,6.9
7,Test,Flares_M,1418,469,48.8
8,Test,Flares_S,5898,815,84.7
